# LightGBM Alzheimer's Disease Biomarker Classification Pipeline

**Heavy, highly-optimized LightGBM model for AD diagnosis prediction using comprehensive biomarker panel**

This notebook implements a complete classification pipeline with:
- Optuna hyperparameter optimization (60 trials)
- Comprehensive evaluation metrics
- Advanced visualizations (confusion matrix, feature importance, SHAP, ROC curves)
- GPU support (automatic detection)
- Focus on diagnostic measures (DXMDES, DXMPTR), advanced biomarkers (Abeta, Tau, GFAP, NFL), and clinical markers (MMSE, DHA, formic acid, lactoferrin, NTP)

---

## 1. Install Dependencies

In [ ]:
!pip install -q lightgbm optuna shap pandas openpyxl scikit-learn matplotlib seaborn imbalanced-learn

## 2. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# LightGBM
import lightgbm as lgb
from lightgbm import LGBMClassifier

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight

# Imbalanced-learn for SMOTE
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Optuna
import optuna
from optuna.samplers import TPESampler

# SHAP
import shap

# Set random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("✓ All libraries imported successfully!")

## 3. Configuration

In [ ]:
# Feature columns - Diagnostic measures and advanced biomarker panel
FEATURE_COLUMNS = [
    # Diagnostic measures
    'DXMDES', 'DXMPTR1', 'DXMPTR2', 'DXMPTR3', 'DXMPTR4', 'DXMPTR5', 'DXMPTR6',
    # Amyloid beta biomarkers
    'Abeta40', 'Abeta42', 'Abeta_ratio',
    # Neuroinflammation and neurodegeneration biomarkers
    'GFAP', 'NFL', 'NfL',
    # Phosphorylated tau biomarkers
    'PTau181', 'npTau217', 'pTau217', 'pTau181', 'pTau217_ratio',
    # Additional biochemical and clinical markers
    'formic_acid_value', 'lactoferrin_value', 'dha_value', 'ntp_value', 'mmse_score'
]

TARGET_COLUMN = 'DIAGNOSIS'
TEST_SIZE = 0.2
N_OPTUNA_TRIALS = 60
CV_FOLDS = 5
RESULTS_DIR = './results_biomarker'

print(f"✓ Configuration set:")
print(f"  - Features: {len(FEATURE_COLUMNS)}")
print(f"  - Test size: {TEST_SIZE*100:.0f}%")
print(f"  - Optuna trials: {N_OPTUNA_TRIALS}")
print(f"  - CV folds: {CV_FOLDS}")

## 4. Upload Data

Upload your Excel file (.xlsx) containing the dataset.

In [ ]:
from google.colab import files

print("Please upload your Excel file...")
uploaded = files.upload()
data_file = list(uploaded.keys())[0]
print(f"\n✓ File uploaded: {data_file}")

## 5. Load and Preprocess Data

In [ ]:
def load_and_preprocess_data(file_path):
    print("=" * 70)
    print("LOADING DATA")
    print("=" * 70)
    
    # Load Excel file
    df = pd.read_excel(file_path)
    print(f"✓ Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
    
    # Select features and target
    required_columns = FEATURE_COLUMNS + [TARGET_COLUMN]
    missing_cols = set(required_columns) - set(df.columns)
    if missing_cols:
        print(f"\n⚠ Warning: Missing columns in dataset: {missing_cols}")
        print("Available columns in dataset:")
        print(df.columns.tolist())
        # Filter out missing columns
        available_features = [col for col in FEATURE_COLUMNS if col in df.columns]
        print(f"\n✓ Using {len(available_features)} available features out of {len(FEATURE_COLUMNS)} specified")
    else:
        available_features = FEATURE_COLUMNS
    
    df_filtered = df[available_features + [TARGET_COLUMN]].copy()
    print(f"✓ Selected {len(available_features)} features + 1 target column")
    
    # Drop rows with missing values
    initial_rows = len(df_filtered)
    df_filtered = df_filtered.dropna()
    dropped_rows = initial_rows - len(df_filtered)
    print(f"✓ Dropped {dropped_rows} rows with missing values ({len(df_filtered)} remaining)")
    
    # ========================================================================
    # FEATURE ENGINEERING: Add biomarker combinations and interactions
    # ========================================================================
    print(f"\n--- Feature Engineering ---")
    
    engineered_features = []
    
    # Aβ42/Aβ40 ratio (if not already present and both components available)
    if 'Abeta_ratio' not in available_features and 'Abeta42' in available_features and 'Abeta40' in available_features:
        df_filtered['Abeta42_40_ratio'] = df_filtered['Abeta42'] / (df_filtered['Abeta40'] + 1e-10)
        engineered_features.append('Abeta42_40_ratio')
        print("✓ Added Aβ42/Aβ40 ratio (gold standard AD biomarker)")
    
    # Tau/Aβ42 ratio (if both available)
    if 'pTau217' in available_features and 'Abeta42' in available_features:
        df_filtered['pTau217_Abeta42_ratio'] = df_filtered['pTau217'] / (df_filtered['Abeta42'] + 1e-10)
        engineered_features.append('pTau217_Abeta42_ratio')
        print("✓ Added pTau217/Aβ42 ratio")
    
    if 'PTau181' in available_features and 'Abeta42' in available_features:
        df_filtered['PTau181_Abeta42_ratio'] = df_filtered['PTau181'] / (df_filtered['Abeta42'] + 1e-10)
        engineered_features.append('PTau181_Abeta42_ratio')
        print("✓ Added PTau181/Aβ42 ratio")
    
    # NFL/GFAP ratio (neurodegeneration vs neuroinflammation)
    nfl_col = 'NFL' if 'NFL' in available_features else ('NfL' if 'NfL' in available_features else None)
    if nfl_col and 'GFAP' in available_features:
        df_filtered['NFL_GFAP_ratio'] = df_filtered[nfl_col] / (df_filtered['GFAP'] + 1e-10)
        engineered_features.append('NFL_GFAP_ratio')
        print("✓ Added NFL/GFAP ratio (neurodegeneration vs neuroinflammation)")
    
    # Composite biomarker burden score
    if 'pTau217' in available_features and 'GFAP' in available_features and nfl_col:
        df_filtered['biomarker_burden'] = (
            df_filtered['pTau217'] +
            df_filtered['GFAP'] +
            df_filtered[nfl_col]
        )
        if 'Abeta42' in available_features:
            df_filtered['biomarker_burden'] -= df_filtered['Abeta42'] * 0.01
        engineered_features.append('biomarker_burden')
        print("✓ Added composite biomarker burden score")
    
    # Diagnostic measure combinations
    dxmptr_cols = [col for col in available_features if col.startswith('DXMPTR')]
    if len(dxmptr_cols) >= 2:
        df_filtered['DXMPTR_mean'] = df_filtered[dxmptr_cols].mean(axis=1)
        df_filtered['DXMPTR_std'] = df_filtered[dxmptr_cols].std(axis=1)
        engineered_features.extend(['DXMPTR_mean', 'DXMPTR_std'])
        print(f"✓ Added DXMPTR aggregations (mean, std) from {len(dxmptr_cols)} measures")
    
    # Update feature list
    all_features = available_features + engineered_features
    
    # Check class distribution
    print(f"\n--- Target Distribution ---")
    class_counts = df_filtered[TARGET_COLUMN].value_counts()
    for label, count in class_counts.items():
        print(f"  {label}: {count} ({count/len(df_filtered)*100:.1f}%)")
    
    # Encode target labels
    X = df_filtered[all_features].values
    y = df_filtered[TARGET_COLUMN].values
    
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    
    print(f"\n✓ Encoded labels: {dict(enumerate(label_encoder.classes_))}")
    
    # Train-test split (stratified)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded,
        test_size=TEST_SIZE,
        random_state=RANDOM_SEED,
        stratify=y_encoded
    )
    
    print(f"\n✓ Train-test split (stratified):")
    print(f"  Training set: {X_train.shape[0]} samples")
    print(f"  Test set: {X_test.shape[0]} samples")
    print(f"✓ Total features: {len(all_features)} ({len(available_features)} original + {len(engineered_features)} engineered)")
    print("=" * 70)
    
    return X_train, X_test, y_train, y_test, label_encoder, all_features

# Load data
X_train, X_test, y_train, y_test, label_encoder, feature_names = load_and_preprocess_data(data_file)
n_classes = len(label_encoder.classes_)

## 6. Hyperparameter Optimization with Optuna

This will take several minutes. Progress bar will be displayed.

In [ ]:
def objective(trial, X_train, y_train, n_classes, class_weights_dict):
    # OPTIMIZED for biomarker-based classification
    params = {
        'objective': 'multiclass',
        'num_class': n_classes,
        'metric': 'multi_logloss',
        'boosting_type': 'gbdt',
        'verbosity': -1,
        'random_state': RANDOM_SEED,
        'device': 'gpu' if os.system('nvidia-smi > /dev/null 2>&1') == 0 else 'cpu',
        'class_weight': class_weights_dict,
        'is_unbalance': True,
        
        # More conservative tree structure
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        
        # Learning parameters
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 300, 2000, step=100),
        
        # Stronger regularization
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
        'min_child_weight': trial.suggest_float('min_child_weight', 1e-3, 1.0, log=True),
        'lambda_l1': trial.suggest_float('lambda_l1', 0.1, 50.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 0.1, 50.0, log=True),
        
        # Sampling
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 0.95),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 0.95),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        
        # Advanced parameters
        'min_split_gain': trial.suggest_float('min_split_gain', 0.01, 1.5),
        'path_smooth': trial.suggest_float('path_smooth', 0.0, 1.0),
    }
    
    # Apply SMOTE for class balancing
    smote = SMOTE(random_state=RANDOM_SEED, k_neighbors=3)
    model = LGBMClassifier(**params)
    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    
    scores = []
    for train_idx, val_idx in cv.split(X_train, y_train):
        X_train_fold, X_val_fold = X_train[train_idx], X_train[val_idx]
        y_train_fold, y_val_fold = y_train[train_idx], y_train[val_idx]
        
        # Apply SMOTE only on training fold
        X_train_resampled, y_train_resampled = smote.fit_resample(X_train_fold, y_train_fold)
        
        # Train and evaluate
        model.fit(X_train_resampled, y_train_resampled)
        score = model.score(X_val_fold, y_val_fold)
        scores.append(score)
    
    return np.mean(scores)

# Compute class weights
print("=" * 70)
print("HYPERPARAMETER OPTIMIZATION (OPTUNA)")
print("=" * 70)

print("\n--- Computing Class Weights ---")
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights_dict = {i: weight for i, weight in enumerate(class_weights)}

print("Class weights (to handle class imbalance):")
for class_idx, weight in class_weights_dict.items():
    print(f"  Class {class_idx}: {weight:.3f}")

print(f"\nRunning {N_OPTUNA_TRIALS} trials with {CV_FOLDS}-fold cross-validation...")
print("This may take several minutes...\n")

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=RANDOM_SEED)
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train, n_classes, class_weights_dict),
    n_trials=N_OPTUNA_TRIALS,
    show_progress_bar=True,
    n_jobs=1
)

print("\n✓ Optimization complete!")
print(f"  Best CV Accuracy: {study.best_value:.4f}")
print(f"  Best Trial: #{study.best_trial.number}")

# Construct best parameters
best_params = study.best_params
best_params.update({
    'objective': 'multiclass',
    'num_class': n_classes,
    'metric': 'multi_logloss',
    'boosting_type': 'gbdt',
    'verbosity': -1,
    'random_state': RANDOM_SEED,
    'device': 'gpu' if os.system('nvidia-smi > /dev/null 2>&1') == 0 else 'cpu',
    'class_weight': class_weights_dict,
    'is_unbalance': True,
})

print("\n--- Best Parameters ---")
for key, value in sorted(best_params.items()):
    if key not in ['objective', 'num_class', 'metric', 'boosting_type', 'verbosity', 'random_state', 'device', 'class_weight', 'is_unbalance']:
        print(f"  {key}: {value}")
print("=" * 70)

best_cv_score = study.best_value

## 7. Train Final Model

In [ ]:
print("=" * 70)
print("TRAINING FINAL MODEL")
print("=" * 70)

# Apply SMOTE to balance classes
print("\n--- Applying SMOTE for Class Balancing ---")
print(f"Original training set size: {X_train.shape[0]}")
print("Original class distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for class_idx, count in zip(unique, counts):
    print(f"  Class {class_idx}: {count} samples")

smote = SMOTE(random_state=RANDOM_SEED, k_neighbors=3)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"\nResampled training set size: {X_train_resampled.shape[0]}")
print("Resampled class distribution:")
unique, counts = np.unique(y_train_resampled, return_counts=True)
for class_idx, count in zip(unique, counts):
    print(f"  Class {class_idx}: {count} samples")

# Train model on resampled data
model = LGBMClassifier(**best_params)
model.fit(X_train_resampled, y_train_resampled)

print("\n✓ Model training complete!")
print("=" * 70)

## 8. Evaluate Model

In [ ]:
def calculate_specificity(y_true, y_pred, n_classes):
    cm = confusion_matrix(y_true, y_pred)
    specificity = {}
    
    for i in range(n_classes):
        tn = np.sum(cm) - (np.sum(cm[i, :]) + np.sum(cm[:, i]) - cm[i, i])
        fp = np.sum(cm[:, i]) - cm[i, i]
        specificity[i] = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    return specificity

# Predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

print("=" * 70)
print("### Performance Metrics")
print("=" * 70)

# Basic metrics
accuracy = accuracy_score(y_test, y_pred)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    y_test, y_pred, average='macro'
)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_test, y_pred, average='weighted'
)

print(f"\nAccuracy: {accuracy:.2f}")
print(f"Macro Avg → Precision: {precision_macro:.2f} | Recall: {recall_macro:.2f} | F1: {f1_macro:.2f}")
print(f"Weighted Avg → Precision: {precision_weighted:.2f} | Recall: {recall_weighted:.2f} | F1: {f1_weighted:.2f}")

# Per-class metrics
print("\n--- Per-Class ---")
precision_per_class, recall_per_class, f1_per_class, _ = precision_recall_fscore_support(
    y_test, y_pred, average=None
)

class_names = label_encoder.classes_
specificity_per_class = calculate_specificity(y_test, y_pred, len(class_names))

for i, class_name in enumerate(class_names):
    print(f"{class_name}: Precision {precision_per_class[i]:.2f} | "
          f"Recall {recall_per_class[i]:.2f} | F1 {f1_per_class[i]:.2f}")

# Confusion Matrix
print("\n--- Confusion Matrix ---")
cm = confusion_matrix(y_test, y_pred)
header = "           " + "".join([f"{name:>7}" for name in class_names])
print(header)
for i, class_name in enumerate(class_names):
    row = f"    {class_name:>2}    " + "".join([f"{cm[i][j]:>7}" for j in range(len(class_names))])
    print(row)

# ROC-AUC Metrics
print("\n--- ROC-AUC Metrics ---")
y_test_bin = label_binarize(y_test, classes=range(len(class_names)))

for i, class_name in enumerate(class_names):
    auc_score = roc_auc_score(y_test_bin[:, i], y_pred_proba[:, i])
    print(f"{class_name} AUC: {auc_score:.2f}")

auc_micro = roc_auc_score(y_test_bin, y_pred_proba, average='micro')
auc_macro = roc_auc_score(y_test_bin, y_pred_proba, average='macro')
auc_weighted = roc_auc_score(y_test_bin, y_pred_proba, average='weighted')

print(f"Micro-average AUC: {auc_micro:.2f}")
print(f"Macro-average AUC: {auc_macro:.2f}")
print(f"Weighted-average AUC: {auc_weighted:.2f}")

# Specificity and Sensitivity
print("\n--- Sensitivity & Specificity per Class ---")
for i, class_name in enumerate(class_names):
    sensitivity = recall_per_class[i]
    specificity = specificity_per_class[i]
    print(f"{class_name}: Sensitivity {sensitivity:.2f} | Specificity {specificity:.2f}")

# Cross-Validation Summary
print("\n--- Cross-Validation Summary ---")
print(f"Mean CV Accuracy: {best_cv_score:.4f}")

print("\n--- Best Hyperparameters ---")
important_params = ['num_leaves', 'max_depth', 'learning_rate', 'n_estimators',
                   'lambda_l1', 'lambda_l2', 'feature_fraction', 'bagging_fraction']
for param in important_params:
    if param in best_params:
        print(f"  {param}: {best_params[param]}")

print("=" * 70)

## 9. Visualizations

In [ ]:
# Create results directory
os.makedirs(RESULTS_DIR, exist_ok=True)

print("=" * 70)
print("GENERATING VISUALIZATIONS")
print("=" * 70)

### 9.1 Confusion Matrix

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved confusion matrix")

### 9.2 Feature Importance

In [ ]:
importance = model.feature_importances_
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importance
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(data=importance_df, x='importance', y='feature', palette='viridis')
plt.title('Feature Importance (LightGBM)', fontsize=16, fontweight='bold')
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'feature_importance.png'), dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved feature importance")

# Save CSV
importance_df.to_csv(os.path.join(RESULTS_DIR, 'feature_importance.csv'), index=False)
print("✓ Saved feature importance CSV")

### 9.3 SHAP Summary Plot

In [ ]:
try:
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    
    plt.figure(figsize=(12, 8))
    if isinstance(shap_values, list):
        shap_values_combined = np.abs(shap_values).mean(axis=0)
        shap.summary_plot(shap_values_combined, X_test,
                        feature_names=feature_names,
                        show=False)
    else:
        shap.summary_plot(shap_values, X_test,
                        feature_names=feature_names,
                        show=False)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'shap_summary.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Saved SHAP summary plot")
except Exception as e:
    print(f"⚠ Warning: Could not generate SHAP plot: {str(e)}")

### 9.4 ROC Curves

In [ ]:
plt.figure(figsize=(10, 8))

colors = ['blue', 'red', 'green', 'orange', 'purple']
for i, (class_name, color) in enumerate(zip(class_names, colors[:len(class_names)])):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_proba[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2,
            label=f'{class_name} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves (One-vs-Rest)', fontsize=16, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'roc_curves.png'), dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved ROC curves")

print("=" * 70)

## 10. Save Results

In [ ]:
print("=" * 70)
print("SAVING RESULTS")
print("=" * 70)

# Save performance metrics
metrics_path = os.path.join(RESULTS_DIR, 'performance_metrics.txt')
with open(metrics_path, 'w') as f:
    f.write("=" * 70 + "\n")
    f.write("### Performance Metrics\n")
    f.write("=" * 70 + "\n\n")
    f.write(f"Accuracy: {accuracy:.2f}\n")
    f.write(f"Macro Avg → Precision: {precision_macro:.2f} | Recall: {recall_macro:.2f} | F1: {f1_macro:.2f}\n")
    f.write(f"Weighted Avg → Precision: {precision_weighted:.2f} | Recall: {recall_weighted:.2f} | F1: {f1_weighted:.2f}\n\n")
    f.write("--- Per-Class ---\n")
    for i, class_name in enumerate(class_names):
        f.write(f"{class_name}: Precision {precision_per_class[i]:.2f} | Recall {recall_per_class[i]:.2f} | F1 {f1_per_class[i]:.2f}\n")
    f.write("\n--- Confusion Matrix ---\n")
    f.write(header + "\n")
    for i, class_name in enumerate(class_names):
        row = f"    {class_name:>2}    " + "".join([f"{cm[i][j]:>7}" for j in range(len(class_names))])
        f.write(row + "\n")
    f.write("\n--- ROC-AUC Metrics ---\n")
    for i, class_name in enumerate(class_names):
        auc_score = roc_auc_score(y_test_bin[:, i], y_pred_proba[:, i])
        f.write(f"{class_name} AUC: {auc_score:.2f}\n")
    f.write(f"Micro-average AUC: {auc_micro:.2f}\n")
    f.write(f"Macro-average AUC: {auc_macro:.2f}\n")
    f.write(f"Weighted-average AUC: {auc_weighted:.2f}\n")
    f.write("\n--- Sensitivity & Specificity per Class ---\n")
    for i, class_name in enumerate(class_names):
        f.write(f"{class_name}: Sensitivity {recall_per_class[i]:.2f} | Specificity {specificity_per_class[i]:.2f}\n")
    f.write("\n--- Cross-Validation Summary ---\n")
    f.write(f"Mean CV Accuracy: {best_cv_score:.4f}\n")
    f.write("\n--- Best Parameters ---\n")
    for key, value in sorted(best_params.items()):
        if key not in ['objective', 'num_class', 'metric', 'boosting_type', 'verbosity', 'random_state', 'device']:
            f.write(f"  {key}: {value}\n")
    f.write("=" * 70 + "\n")

print(f"✓ Saved performance metrics: {metrics_path}")

# Save classification report
report_path = os.path.join(RESULTS_DIR, 'classification_report.txt')
report = classification_report(y_test, y_pred, target_names=class_names)
with open(report_path, 'w') as f:
    f.write("Classification Report\n")
    f.write("=" * 70 + "\n\n")
    f.write(report)

print(f"✓ Saved classification report: {report_path}")
print("=" * 70)

## 11. Download Results

Download all results as a zip file.

In [ ]:
from google.colab import files
import shutil

# Create zip file
shutil.make_archive('results_biomarker', 'zip', RESULTS_DIR)

# Download
files.download('results_biomarker.zip')
print("✓ Results downloaded!")

## 12. Final Summary

In [ ]:
print("\n" + "=" * 70)
print(" PIPELINE COMPLETE!")
print("=" * 70)
print(f"\n✓ All results saved to: {RESULTS_DIR}/")
print(f"✓ Test Accuracy: {accuracy:.2f}")
print(f"✓ Macro F1-Score: {f1_macro:.2f}")
print(f"✓ Macro AUC: {auc_macro:.2f}")
print(f"✓ Best CV Accuracy: {best_cv_score:.4f}")
print("\n" + "=" * 70)
print("Results files:")
print("  - performance_metrics.txt")
print("  - classification_report.txt")
print("  - feature_importance.csv")
print("  - confusion_matrix.png")
print("  - feature_importance.png")
print("  - shap_summary.png")
print("  - roc_curves.png")
print("=" * 70 + "\n")